In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

def calculate_average_pixel_value(frames):
    return [np.mean(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)) for frame in frames]

def extract_frames_and_length(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    vid_len = frame_count / fps if fps > 0 else 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    
    cap.release()
    return frames, vid_len

# Directories
video_folder = 'cropped_videos'  # Replace with actual video folder path
output_folder = 'vid_binary_plots'
os.makedirs(output_folder, exist_ok=True)

# Process all videos
for filename in os.listdir(video_folder):
    if filename.endswith('.mp4'):
        video_path = os.path.join(video_folder, filename)
        video_id = filename.split('_')[-1].split('.')[0]  # Extract ID from filename
        
        frames, vid_len = extract_frames_and_length(video_path)
        average_values = calculate_average_pixel_value(frames)
        
        threshold = max(average_values) - (max(average_values) - min(average_values)) * 0.5
        binary_values = [1 if val > threshold else 0 for val in average_values]
        frame_indices = np.linspace(0, vid_len, len(binary_values))
        
        # Save binary values for matching algorithm
        np.savetxt(os.path.join(output_folder, f'vid_ID_{video_id}.txt'), binary_values, fmt='%d')
        
        # Plot and save
        plt.figure()
        plt.step(frame_indices, binary_values, where='mid')
        plt.xlabel("Time (seconds)")
        plt.ylabel("Binary Pixel Value (0 or 1)")
        plt.title(f"Thresholded Pixel Intensity - Video ID {video_id}")
        plt.ylim(-0.1, 1.1)
        plt.savefig(os.path.join(output_folder, f'vid_ID_{video_id}.png'))
        plt.close()


## Audio

In [3]:
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import os

def process_audio_file(audio_path, output_folder, threshold=0.5):
    # Load audio file
    y, sr = librosa.load(audio_path, sr=None)
    
    # Apply thresholding: set values to 1 if greater than threshold, else 0
    y_binary = np.where(np.abs(y) > threshold, 1, 0)
    
    # Create time axis
    time = np.linspace(0, len(y) / sr, num=len(y))
    
    # Extract ID from filename
    audio_id = os.path.splitext(os.path.basename(audio_path))[0].split('_')[-1]
    
    # Save binary values for matching algorithm
    np.savetxt(os.path.join(output_folder, f'audio_ID_{audio_id}.txt'), y_binary, fmt='%d')
    
    # Plot and save
    plt.figure(figsize=(12, 5))
    plt.step(time, y_binary, where="mid", label="Binary Signal", color="r")
    plt.xlabel("Time (seconds)")
    plt.ylabel("Binary Value (0 or 1)")
    plt.title(f"Binary Audio Signal - Audio ID {audio_id}")
    plt.legend()
    plt.grid()
    plt.ylim(-0.1, 1.1)
    plt.savefig(os.path.join(output_folder, f'audio_ID_{audio_id}.png'))
    plt.close()

# Directories
audio_folder = 'dataset/audio_only'  # Replace with actual audio folder path
output_folder = 'binary_audio_plots'
os.makedirs(output_folder, exist_ok=True)

# Process all audio files
for filename in os.listdir(audio_folder):
    if filename.endswith('.wav'):
        audio_path = os.path.join(audio_folder, filename)
        process_audio_file(audio_path, output_folder, threshold=0.49)
